# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = "task137"
CH = 10
H = W = 30
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
ONNX_PATH = Path(f"{TASK_ID}.onnx")
SUBMISSION_PATH = Path("submission.zip")

In [6]:
LOCAL_TASK_JSON = Path("/mnt/data/task137.json")
if LOCAL_TASK_JSON.exists():
    task = json.loads(LOCAL_TASK_JSON.read_text())
elif Path("task137.json").exists():
    task = json.loads(Path("task137.json").read_text())
else:
    task = None

def grid_to_tensor(grid):
    arr = np.asarray(grid, dtype=np.int64)
    x = np.zeros((1, CH, H, W), dtype=np.float32)
    h, w = arr.shape
    for c in range(CH):
        x[0, c, :h, :w] = (arr == c)
    return x

def expected_tensor(grid):
    arr = np.asarray(grid, dtype=np.int64)
    y = np.zeros((1, CH, H, W), dtype=np.float32)
    h, w = arr.shape
    for c in range(CH):
        y[0, c, :h, :w] = (arr == c)
    return y

In [7]:
class Task137CanvasMaskModel(nn.Module):
    def __init__(self):
        super().__init__()
        rr = torch.arange(H, dtype=torch.float32).view(1, H, 1).expand(1, H, W)
        cc = torch.arange(W, dtype=torch.float32).view(1, 1, W).expand(1, H, W)
        colors = torch.arange(CH, dtype=torch.float32).view(1, CH, 1, 1)
        self.register_buffer("rr", rr)
        self.register_buffer("cc", cc)
        self.register_buffer("colors", colors)

    def forward(self, x):
        # Inside true canvas: exactly one channel is 1. Outside true canvas: all channels are 0.
        active = (torch.sum(x, dim=1) > 0.5).float()
        fg = torch.sum(x[:, 1:, :, :], dim=1)
        n = torch.sum(fg).reshape(1, 1, 1)
        n_safe = torch.maximum(n, torch.tensor(1.0, dtype=torch.float32, device=x.device))
        r0 = (torch.sum(fg * self.rr) / n_safe).reshape(1, 1, 1)
        c0 = (torch.sum(fg * self.cc) / n_safe).reshape(1, 1, 1)
        color_val = (torch.sum(x * self.colors) / n_safe).reshape(1, 1, 1, 1)

        dr = torch.abs(self.rr - r0)
        dc = torch.abs(self.cc - c0)
        step_r = torch.amax(dr * fg, dim=(1, 2), keepdim=True)
        step_c = torch.amax(dc * fg, dim=(1, 2), keepdim=True)
        step = torch.maximum(step_r, step_c)
        cheb = torch.maximum(dr, dc)

        draw = torch.zeros_like(active)
        for k in range(31):
            draw = torch.maximum(draw, (torch.abs(cheb - step * float(k)) < 0.25).float())
        draw = draw * active

        outs = []
        for c in range(CH):
            ch = x[:, c, :, :]
            if c == 0:
                ch = ch * (1.0 - draw)
            else:
                color_flag = (torch.abs(color_val - float(c)) < 0.25).float().reshape(1, 1, 1)
                ch = torch.maximum(ch, draw * color_flag)
            outs.append(ch)
        return torch.stack(outs, dim=1)

model = Task137CanvasMaskModel().eval()

In [8]:
dummy = np.zeros((1, CH, H, W), dtype=np.float32)
dummy[0, 0, :28, :28] = 1.0
for r, c in [(12, 7), (18, 13), (24, 19)]:
    dummy[0, 0, r, c] = 0.0
    dummy[0, 4, r, c] = 1.0

torch.onnx.export(
    model,
    torch.tensor(dummy),
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

m = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(m)
m = shape_inference.infer_shapes(m)
onnx.save(m, str(ONNX_PATH))
print("saved", ONNX_PATH, "bytes", ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/4204067472.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_16/4262663143.py:16: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  n_safe = torch.maximum(n, torch.tensor(1.0, dtype=torch.float32, device=x.device))


saved task137.onnx bytes 58529


In [9]:
m = onnx.load(str(ONNX_PATH))
ops = sorted({node.op_type for node in m.graph.node})
print("ops:", ops)
print("forbidden:", sorted(set(ops) & FORBIDDEN_OPS))
print("empty optional inputs:", [(n.name, n.op_type) for n in m.graph.node for i in n.input if i == ""])

def dims(v):
    return [d.dim_value if d.HasField("dim_value") else None for d in v.type.tensor_type.shape.dim]
print("input shape:", dims(m.graph.input[0]))
print("output shape:", dims(m.graph.output[0]))
bad = []
for v in list(m.graph.input) + list(m.graph.output) + list(m.graph.value_info):
    ds = dims(v)
    if not ds or any(d in (None, 0) for d in ds):
        bad.append((v.name, ds))
print("bad static shapes count:", len(bad))

ops: ['Abs', 'Cast', 'Concat', 'Constant', 'Div', 'Gather', 'Greater', 'Less', 'Max', 'Mul', 'ReduceMax', 'ReduceSum', 'Reshape', 'Slice', 'Sub', 'Unsqueeze']
forbidden: []
empty optional inputs: []
input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
bad static shapes count: 97


In [10]:
if task is not None:
    sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
    for split in ["train", "test", "arc-gen"]:
        ok = total = 0
        outside_zero_ok = 0
        for ex in task.get(split, []):
            y = sess.run(None, {sess.get_inputs()[0].name: grid_to_tensor(ex["input"])})[0]
            e = expected_tensor(ex["output"])
            ok += bool(np.allclose(y, e, atol=1e-5))
            arr = np.asarray(ex["input"])
            h, w = arr.shape
            mask = np.ones((H, W), dtype=bool)
            mask[:h, :w] = False
            outside_zero_ok += bool(np.allclose(y[0, :, mask], 0.0, atol=1e-6))
            total += 1
        print(f"{split}_tensor_exact_zero_padded: {ok}/{total}; outside_zero: {outside_zero_ok}/{total}")

In [11]:
with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")
print("wrote", SUBMISSION_PATH)

wrote submission.zip
